# 07-7. 로컬 웹 보안 점검 프로젝트 예제

## Goal

- 여러 점검 결과를 하나의 보고서로 통합합니다.
- 경고와 실패를 별도 집계합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

가짜 세션으로 고정 응답을 사용하므로 서버 실행과 네트워크가 필요하지 않습니다.


## Steps

### 점검 파이프라인 통합

연결·응답 계약·보안 헤더·리다이렉트 결과를 같은 구조로 모읍니다.


In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class FakeResponse:
    status: int
    headers: dict[str, str]
    body: dict


RESPONSES = {
    "/health": FakeResponse(200, {"Content-Type": "application/json"}, {"status": "ok"}),
    "/headers": FakeResponse(200, {"Content-Type": "application/json"}, {}),
    "/redirect": FakeResponse(302, {"Location": "/health"}, {}),
}


def run_checks(responses):
    checks = []
    health = responses["/health"]
    checks.append({"check": "health", "status": "pass" if health.status == 200 and health.body.get("status") == "ok" else "fail"})
    required = {"Content-Security-Policy", "X-Content-Type-Options", "Referrer-Policy"}
    missing = sorted(required - responses["/headers"].headers.keys())
    checks.append({"check": "headers", "status": "warning" if missing else "pass", "missing": missing})
    redirect = responses["/redirect"]
    checks.append({"check": "redirect", "status": "pass" if redirect.headers.get("Location") == "/health" else "fail"})
    summary = {status: sum(item["status"] == status for item in checks) for status in ("pass", "warning", "fail")}
    return {"scope": "합성 로컬 응답", "summary": summary, "checks": checks}


report = run_checks(RESPONSES)
print(report)


{'scope': '합성 로컬 응답', 'summary': {'pass': 2, 'warning': 1, 'fail': 0}, 'checks': [{'check': 'health', 'status': 'pass'}, {'check': 'headers', 'status': 'warning', 'missing': ['Content-Security-Policy', 'Referrer-Policy', 'X-Content-Type-Options']}, {'check': 'redirect', 'status': 'pass'}]}


## Checks

의도한 헤더 경고 한 건과 실패 없음 상태를 확인합니다.


In [2]:
assert report["summary"] == {"pass": 2, "warning": 1, "fail": 0}
assert report["checks"][1]["missing"]
assert report["scope"] == "합성 로컬 응답"
print("프로젝트 보고서 검사 통과")


프로젝트 보고서 검사 통과


## Next Steps

실제 프로젝트는 터미널에서 `training_server.py`와 `security_validator.py`를 별도로 실행합니다.
